In [340]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from utils.evaluation import *
from utils.preprocessing import extract_sequence_embeddings

In [341]:
df_test = pd.read_csv("lemmatized\\test_lemmatized.csv")
df_expert_180 = pd.read_csv("lemmatized\\expert_core_180_lemmatized.csv")

## Preprocessing

In [342]:
n_per_class_teacher = 30
random_seed = 42

teacher_dfs = []
student_dfs = []

for label in [0, 1, 2]:
    class_data = df_expert_180[df_expert_180['label'] == label].copy()
    class_data = class_data.sample(frac=1, random_state=random_seed).reset_index(drop=True)

    teacher_part = class_data.iloc[:n_per_class_teacher]
    student_part = class_data.iloc[n_per_class_teacher:]

    teacher_dfs.append(teacher_part)
    student_dfs.append(student_part)

df_teacher_train = pd.concat(teacher_dfs).sample(frac=1, random_state=random_seed).reset_index(drop=True)
df_student_train = pd.concat(student_dfs).sample(frac=1, random_state=random_seed).reset_index(drop=True)

In [343]:
def prepare_dataset(df, text_col='text_lemmatized', label_col='label'):
    df[text_col] = df[text_col].astype(str)

    mask = (df[text_col] != 'nan') & (df[text_col].str.strip() != '')
    df = df[mask].copy()

    dataset = Dataset.from_pandas(df[[text_col, label_col]])
    dataset = dataset.rename_column(text_col, "text")
    dataset = dataset.rename_column(label_col, "labels")
    return dataset

In [344]:
dataset_test = prepare_dataset(df_test)
dataset_teacher_train = prepare_dataset(df_teacher_train)
dataset_student_train = prepare_dataset(df_student_train)

In [345]:
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [346]:
def tokenize_function(examples):
    texts = examples["text"]

    cleaned_texts = []
    for t in texts:
        if t is None or (isinstance(t, float) and t != t):
            cleaned_texts.append("")
        elif not isinstance(t, str):
            cleaned_texts.append(str(t))
        else:
            cleaned_texts.append(t)

    return tokenizer(
        cleaned_texts,
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [347]:
tokenized_test = dataset_test.map(tokenize_function, batched=True)
tokenized_teacher_train = dataset_teacher_train.map(tokenize_function, batched=True)
tokenized_student_train = dataset_student_train.map(tokenize_function, batched=True)

columns_to_keep = ['input_ids', 'attention_mask', 'labels']
tokenized_teacher_train = tokenized_teacher_train.remove_columns(['text'])
tokenized_student_train = tokenized_student_train.remove_columns(['text'])

Map: 100%|██████████| 90/90 [00:00<00:00, 13647.90 examples/s]


## Training Teacher

In [348]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [349]:
def train_model(train_ds, eval_ds, name, epochs, batch_size, lr):
    print(f"\n{'='*50}")
    print(f"Обучение: {name}")
    print(f"{'='*50}")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3, problem_type="single_label_classification"
    )

    args = TrainingArguments(
        output_dir=f"./results_{name}",
        eval_strategy="epoch" if eval_ds else "no",
        save_strategy="epoch" if eval_ds else "no",
        logging_steps=50,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        weight_decay=0.01,
        load_best_model_at_end=True if eval_ds else False,
        metric_for_best_model="f1_macro" if eval_ds else None,
        report_to="none",
        fp16=True if device == "cuda" else False,
        seed=42,
        disable_tqdm=False
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        compute_metrics=compute_metrics
    )

    trainer.train()
    return trainer

In [350]:
trainer_expert_45 = train_model(
    tokenized_teacher_train, None, "expert_few_shot",
    epochs=10, batch_size=5, lr=1e-4
)


Обучение: expert_few_shot


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1508.75it/s, Materializing param=bert.pooler.dense.weight]                              
BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/archi

Step,Training Loss
50,0.838445
100,0.201533
150,0.041131


## Training Student

In [478]:
class MultiScaleBERTCNN(nn.Module):
    def __init__(self, embed_dim=312, kernel_sizes=[2, 3, 4, 5, 6], num_filters=64, num_classes=3):
        super(MultiScaleBERTCNN, self).__init__()

        # Вход: [Batch, SeqLen=128, EmbedDim=312]
        # После permute: [Batch, Channels=312, SeqLen=128]

        # Создаем список сверток разных размеров (как в фишинг-проекте)
        # Каждая свертка скользит по последовательности токенов
        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.ConvTranspose1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k, padding=k//2), # Padding для сохранения длины
                nn.ReLU(),
                nn.AdaptiveMaxPool1d(1)
            ) for k in kernel_sizes
        ])

        # Объединение признаков от всех сверток
        combined_dim = num_filters * len(kernel_sizes)

        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        # x shape: [Batch, 128, 312]
        x = x.permute(0, 2, 1)  # [Batch, 312, 128]

        conv_outputs = [conv(x).squeeze(-1) for conv in self.convs]

        # Конкатенируем по каналу признаков: [Batch, num_filters * len(kernels)]
        combined = torch.cat(conv_outputs, dim=1)

        return self.classifier(combined)

In [352]:
teacher_model = trainer_expert_45.model
teacher_model.eval()
teacher_model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(83828, 312, padding_idx=0)
      (position_embeddings): Embedding(2048, 312)
      (token_type_embeddings): Embedding(2, 312)
      (LayerNorm): LayerNorm((312,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-2): 3 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=312, out_features=312, bias=True)
              (key): Linear(in_features=312, out_features=312, bias=True)
              (value): Linear(in_features=312, out_features=312, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=312, out_features=312, bias=True)
              (LayerNorm): LayerNorm((312,), eps=1e-12, 

In [353]:
X_student, y_student = extract_sequence_embeddings(trainer_expert_45, tokenized_student_train, device, batch_size=15)

Extracting embeddings: 100%|██████████| 6/6 [00:00<00:00, 254.41batch/s]


In [354]:
X_test, y_test = extract_sequence_embeddings(trainer_expert_45, tokenized_test, device, batch_size=64)

Extracting embeddings: 100%|██████████| 235/235 [00:01<00:00, 126.15batch/s]


In [355]:
train_dataset = TensorDataset(X_student, y_student)
train_loader = DataLoader(train_dataset, batch_size=15, shuffle=True)

In [ ]:
student_model = MultiScaleBERTCNN(
    embed_dim=312,
    kernel_sizes=[2, 3, 4, 5, 6],
    num_filters=12,
    num_classes=3
).to(device)

total_params = sum(p.numel() for p in student_model.parameters())
print(f"\nMulti-Scale Student CNN Parameters: {total_params:,}")


Multi-Scale Student CNN Parameters: 138,743


In [504]:
num_epochs = 200

optimizer_student = optim.SGD(
    student_model.parameters(),
    lr=0.02,
    momentum=0.6,
    weight_decay=0.01
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer_student, T_max=num_epochs, eta_min=1e-6)

criterion_student = nn.CrossEntropyLoss()

student_model.train()

for epoch in range(num_epochs):
    epoch_loss = 0
    for batch in train_loader:
        inputs, labels = [b.to(device) for b in batch]

        optimizer_student.zero_grad()
        outputs = student_model(inputs)
        loss = criterion_student(outputs, labels)
        loss.backward()
        optimizer_student.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)

    scheduler.step(avg_loss)

    if (epoch + 1) % 10 == 0:
        current_lr = optimizer_student.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}, LR: {current_lr:.6f}")

C:\Users\Artem\AppData\Local\Temp\ipykernel_39960\2816884449.py:31: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  scheduler.step(avg_loss)


Epoch 10/200, Loss: 0.3010, LR: 0.020000
Epoch 20/200, Loss: 0.0807, LR: 0.020000
Epoch 30/200, Loss: 0.0238, LR: 0.020000
Epoch 40/200, Loss: 0.0079, LR: 0.020000
Epoch 50/200, Loss: 0.0129, LR: 0.020000
Epoch 60/200, Loss: 0.0113, LR: 0.020000
Epoch 70/200, Loss: 0.0033, LR: 0.020000
Epoch 80/200, Loss: 0.0021, LR: 0.020000
Epoch 90/200, Loss: 0.0051, LR: 0.020000
Epoch 100/200, Loss: 0.0045, LR: 0.020000
Epoch 110/200, Loss: 0.0070, LR: 0.020000
Epoch 120/200, Loss: 0.0042, LR: 0.020000
Epoch 130/200, Loss: 0.0072, LR: 0.020000
Epoch 140/200, Loss: 0.0079, LR: 0.020000
Epoch 150/200, Loss: 0.0045, LR: 0.020000
Epoch 160/200, Loss: 0.0070, LR: 0.020000
Epoch 170/200, Loss: 0.0050, LR: 0.020000
Epoch 180/200, Loss: 0.0038, LR: 0.020000
Epoch 190/200, Loss: 0.0092, LR: 0.020000
Epoch 200/200, Loss: 0.0049, LR: 0.020000


In [505]:
student_model.eval()
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=64)

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        inputs, labels = [b.to(device) for b in batch]
        outputs = student_model(inputs)
        preds = torch.argmax(outputs, dim=-1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
f1_macro = f1_score(all_labels, all_preds, average='macro')
f1_weighted = f1_score(all_labels, all_preds, average='weighted')

print(f"\nResults: Distillation (RuBERT-Teacher 90ex + CNN-Student 90ex)")
print(f"   Accuracy:     {acc:.4f}")
print(f"   F1 Macro:     {f1_macro:.4f}")
print(f"   F1 Weighted:  {f1_weighted:.4f}")


Results: Distillation (RuBERT-Teacher 90ex + CNN-Student 90ex)
   Accuracy:     0.5537
   F1 Macro:     0.5567
   F1 Weighted:  0.5567
